# UNSW-NB15: Data Preparation and Two-Tier Models

Preparing network flows for a two-tier classifier, then training and evaluating a logistic regression baseline for **normal vs. attack** detection.

1. Load and inspect the dataset.
2. Remove duplicates and create stratified train, validation, and test splits.
3. Prepare Tier 1 and Tier 2 datasets and fit feature preprocessing.
4. Train the Tier 1 baseline and evaluate it on validation data.
5. Tune the decision threshold and inspect recall by attack family.
6. Compare a LightGBM Tier 1 model and train a Tier 2 attack-family classifier.

Run cells from top to bottom with the working directory set to `notebooks/`. The input is `../data/UNSW-NB15/UNSW-NB15.csv`; the fitted preprocessor is saved to `preprocessor.joblib` in the working directory. Final test evaluation and end-to-end two-tier evaluation are left for later work.

## 1. Setup and Data Loading

In [37]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("../data/UNSW-NB15/UNSW-NB15.csv")
PREPROCESSOR_PATH = Path("preprocessor.joblib")
RANDOM_STATE = 42
TARGET_FPR = 0.01

In [38]:
df = pd.read_csv(DATA_PATH)
df.head()

,proto,state,dur,sbytes,dbytes,sttl,dttl,sloss,dloss,service,...,ct_flw_http_mthd,is_ftp_login,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,Udp,Con,0.036133,528,304,31,29,0,0,-,...,0,0,0,2,4,1,1,2,Normal,0
1,Udp,Con,0.001119,146,178,31,29,0,0,Dns,...,0,0,0,12,8,2,1,1,Normal,0
2,Udp,Con,0.001209,132,164,31,29,0,0,Dns,...,0,0,0,6,9,1,1,1,Normal,0
3,Udp,Con,0.001169,146,178,31,29,0,0,Dns,...,0,0,0,7,9,1,1,1,Normal,0
4,Udp,Con,0.078339,568,312,31,29,0,0,-,...,0,0,0,2,4,1,1,2,Normal,0


## 2. Explore the Raw Data

Inspect the schema, target balance, missing values, and duplicate rows before modifying the dataset.

In [39]:
print("Rows and columns:", df.shape)
df.info()

Rows and columns: (2540043, 38)
<class 'pandas.DataFrame'>
RangeIndex: 2540043 entries, 0 to 2540042
Data columns (total 38 columns):
 #   Column            Dtype  
---  ------            -----  
 0   proto             str    
 1   state             str    
 2   dur               float64
 3   sbytes            int64  
 4   dbytes            int64  
 5   sttl              int64  
 6   dttl              int64  
 7   sloss             int64  
 8   dloss             int64  
 9   service           str    
 10  Sload             float64
 11  Spkts             int64  
 12  Dpkts             int64  
 13  swin              int64  
 14  dwin              int64  
 15  stcpb             int64  
 16  dtcpb             int64  
 17  smeansz           int64  
 18  dmeansz           int64  
 19  trans_depth       int64  
 20  res_bdy_len       int64  
 21  Sjit              float64
 22  Djit              float64
 23  Sintpkt           float64
 24  Dintpkt           float64
 25  tcprtt            float6

In [40]:
df.columns.tolist()

['proto',
 'state',
 'dur',
 'sbytes',
 'dbytes',
 'sttl',
 'dttl',
 'sloss',
 'dloss',
 'service',
 'Sload',
 'Spkts',
 'Dpkts',
 'swin',
 'dwin',
 'stcpb',
 'dtcpb',
 'smeansz',
 'dmeansz',
 'trans_depth',
 'res_bdy_len',
 'Sjit',
 'Djit',
 'Sintpkt',
 'Dintpkt',
 'tcprtt',
 'is_sm_ips_ports',
 'ct_state_ttl',
 'ct_flw_http_mthd',
 'is_ftp_login',
 'ct_ftp_cmd',
 'ct_srv_src',
 'ct_srv_dst',
 'ct_src_dport_ltm',
 'ct_dst_sport_ltm',
 'ct_dst_src_ltm',
 'attack_cat',
 'Label']

In [41]:
print("Binary label counts:")
print(df["Label"].value_counts())

print("\nAttack category distribution:")
category_counts = df["attack_cat"].value_counts()
category_distribution = pd.DataFrame({
    "Count": category_counts,
    "Percent": df["attack_cat"].value_counts(normalize=True) * 100,
})
category_distribution

Binary label counts:
Label
0    2218760
1     321283
Name: count, dtype: int64

Attack category distribution:


,Count,Percent
attack_cat,,
Normal,2218760,87.351277
Generic,215481,8.483360
Exploits,44525,1.752923
Fuzzers,24246,0.954551
Dos,16353,0.643808
Reconnaissance,13987,0.550660
Analysis,2677,0.105392
Backdoors,2329,0.091691
Shellcode,1511,0.059487


In [42]:
print("Missing values per column:")
print(df.isna().sum().sort_values(ascending=False))
print("\nDuplicate rows:", df.duplicated().sum())

Missing values per column:
proto               0
ct_flw_http_mthd    0
Sjit                0
Djit                0
Sintpkt             0
Dintpkt             0
tcprtt              0
is_sm_ips_ports     0
ct_state_ttl        0
is_ftp_login        0
state               0
ct_ftp_cmd          0
ct_srv_src          0
ct_srv_dst          0
ct_src_dport_ltm    0
ct_dst_sport_ltm    0
ct_dst_src_ltm      0
attack_cat          0
res_bdy_len         0
trans_depth         0
dmeansz             0
smeansz             0
dur                 0
sbytes              0
dbytes              0
sttl                0
dttl                0
sloss               0
dloss               0
service             0
Sload               0
Spkts               0
Dpkts               0
swin                0
dwin                0
stcpb               0
dtcpb               0
Label               0
dtype: int64

Duplicate rows: 571625


## 3. Remove Duplicates and Inspect Targets

Remove exact duplicate rows before splitting so identical flows cannot appear in multiple splits. Cross-check binary labels against attack categories after deduplication.

In [43]:
df = df.drop_duplicates().reset_index(drop=True)

print("After deduplication:", df.shape)
print("Duplicates:", df.duplicated().sum())

After deduplication: (1968418, 38)
Duplicates: 0


In [44]:
print("\nBinary labels:")
print(df["Label"].value_counts())

print("\nAttack categories:")
print(df["attack_cat"].value_counts())

print("\nLabel consistency:")
print(pd.crosstab(df["attack_cat"], df["Label"]))


Binary labels:
Label
0    1885530
1      82888
Name: count, dtype: int64

Attack categories:
attack_cat
Normal            1885530
Exploits            27187
Fuzzers             20347
Generic             15895
Reconnaissance       9076
Dos                  5262
Analysis             1809
Backdoors            1710
Shellcode            1432
Worms                 170
Name: count, dtype: int64

Label consistency:
Label                 0      1
attack_cat                    
Analysis              0   1809
Backdoors             0   1710
Dos                   0   5262
Exploits              0  27187
Fuzzers               0  20347
Generic               0  15895
Normal          1885530      0
Reconnaissance        0   9076
Shellcode             0   1432
Worms                 0    170


## 4. Create and Verify the Data Splits

Use 70% of the deduplicated data for training, 15% for validation, and 15% for testing. Stratify both splits by `attack_cat` to preserve attack-family proportions. Use validation data for model and threshold analysis; reserve test data for final evaluation.

In [45]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=df["attack_cat"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["attack_cat"],
)

In [46]:
# Distribution of attack categories in train, validation, and test sets

print("\nTrain:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

distribution = pd.DataFrame({
    "Train": train_df["attack_cat"].value_counts(),
    "Validation": val_df["attack_cat"].value_counts(),
    "Test": test_df["attack_cat"].value_counts()
})

print(distribution)


Train: (1377892, 38)
Validation: (295263, 38)
Test: (295263, 38)
                  Train  Validation    Test
attack_cat                                 
Normal          1319871      282829  282830
Exploits          19031        4078    4078
Fuzzers           14243        3052    3052
Generic           11127        2384    2384
Reconnaissance     6353        1361    1362
Dos                3683         789     790
Analysis           1266         272     271
Backdoors          1197         257     256
Shellcode          1002         215     215
Worms               119          26      25


In [47]:
print("Training attack percentage:", train_df["Label"].mean() * 100)
print("Validation attack percentage:", val_df["Label"].mean() * 100)

Training attack percentage: 4.210852519645951
Validation attack percentage: 4.211160897233992


## 5. Prepare Features and Targets

Exclude both target columns from the features. Tier 1 uses every flow and the binary `Label`; Tier 2 uses attack flows only and predicts `attack_cat`.

### 5.1 Tier 1: Normal vs. Attack

In [48]:
feature_columns = [
    col for col in df.columns if col not in ["Label", "attack_cat"]
]

X_train = train_df[feature_columns]
y_train = train_df["Label"]

X_val = val_df[feature_columns]
y_val = val_df["Label"]

X_test = test_df[feature_columns]
y_test = test_df["Label"]

print("Train features / target:", X_train.shape, y_train.shape)
print("Validation features / target:", X_val.shape, y_val.shape)
print("Test features / target:", X_test.shape, y_test.shape)

Train features / target: (1377892, 36) (1377892,)
Validation features / target: (295263, 36) (295263,)
Test features / target: (295263, 36) (295263,)


### 5.2 Tier 2: Attack Family

In [49]:
attack_train = train_df[train_df["Label"] == 1].copy()
attack_val = val_df[val_df["Label"] == 1].copy()
attack_test = test_df[test_df["Label"] == 1].copy()

X_train_tier2 = attack_train[feature_columns]
y_train_tier2 = attack_train["attack_cat"]

X_val_tier2 = attack_val[feature_columns]
y_val_tier2 = attack_val["attack_cat"]

X_test_tier2 = attack_test[feature_columns]
y_test_tier2 = attack_test["attack_cat"]

## 6. Preprocess Features

One-hot encode `proto`, `state`, and `service`, and standardize the remaining features. Fit the preprocessor on Tier 1 training data only, then reuse it to transform validation and test data. The Tier 2 datasets above remain untransformed.

In [50]:
categorical_features = ["proto", "state", "service"]
numerical_features = [
    col for col in X_train.columns if col not in categorical_features
]

print("Categorical features:", categorical_features)
print("Number of numerical features:", len(numerical_features))

Categorical features: ['proto', 'state', 'service']
Number of numerical features: 33


In [51]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=True),
            categorical_features,
        ),
        ("numerical", StandardScaler(), numerical_features),
    ]
)

In [52]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Original training shape: (1377892, 36)
Processed training shape: (1377892, 197)
Processed validation shape: (295263, 197)
Processed test shape: (295263, 197)


In [53]:
# Get the feature names after preprocessing

feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))
print(feature_names[:50])

Number of processed features: 197
['categorical__proto_3Pc' 'categorical__proto_A/N'
 'categorical__proto_Aes-Sp3-D' 'categorical__proto_Any'
 'categorical__proto_Argus' 'categorical__proto_Aris'
 'categorical__proto_Arp' 'categorical__proto_Ax.25'
 'categorical__proto_Bbn-Rcc' 'categorical__proto_Bna'
 'categorical__proto_Br-Sat-Mon' 'categorical__proto_Cbt'
 'categorical__proto_Cftp' 'categorical__proto_Chaos'
 'categorical__proto_Compaq-Peer' 'categorical__proto_Cphb'
 'categorical__proto_Cpnx' 'categorical__proto_Crtp'
 'categorical__proto_Crudp' 'categorical__proto_Dcn'
 'categorical__proto_Ddp' 'categorical__proto_Ddx'
 'categorical__proto_Dgp' 'categorical__proto_Egp'
 'categorical__proto_Eigrp' 'categorical__proto_Emcon'
 'categorical__proto_Encap' 'categorical__proto_Esp'
 'categorical__proto_Etherip' 'categorical__proto_Fc'
 'categorical__proto_Fire' 'categorical__proto_Ggp'
 'categorical__proto_Gmtp' 'categorical__proto_Gre'
 'categorical__proto_Hmp' 'categorical__proto_I-Nl

### 6.1 Save the Fitted Preprocessor

In [54]:
joblib.dump(preprocessor, PREPROCESSOR_PATH)

['preprocessor.joblib']

## 7. Train the Tier 1 Baseline

Fit logistic regression with balanced class weights. Generate validation predictions and attack probabilities for evaluation and threshold selection.

In [55]:
tier1_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=RANDOM_STATE,
)
tier1_model.fit(X_train_processed, y_train)

y_val_pred = tier1_model.predict(X_val_processed)
y_val_prob = tier1_model.predict_proba(X_val_processed)[:, 1]

## 8. Evaluate the Default Decision Threshold

Report validation classification metrics and the confusion matrix at the default decision threshold. Average precision summarizes the precision–recall curve using the predicted probabilities.

In [56]:
print(classification_report(
    y_val,
    y_val_pred,
    target_names=["Normal", "Attack"],
))

cm = confusion_matrix(y_val, y_val_pred)
print("Confusion matrix (rows: actual, columns: predicted; Normal, Attack):")
print(cm)

              precision    recall  f1-score   support

      Normal       1.00      0.99      0.99    282829
      Attack       0.75      1.00      0.86     12434

    accuracy                           0.99    295263
   macro avg       0.87      0.99      0.92    295263
weighted avg       0.99      0.99      0.99    295263

Confusion matrix (rows: actual, columns: predicted; Normal, Attack):
[[278674   4155]
 [     4  12430]]


In [57]:
# Calculate evaluation metrics

tn, fp, fn, tp = cm.ravel()

precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)

false_positive_rate = fp / (fp + tn)

pr_auc = average_precision_score(
    y_val,
    y_val_prob
)

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("False Positive Rate:", false_positive_rate)
print("PR-AUC:", pr_auc)

Precision: 0.7494724148326801
Recall: 0.9996783014315587
F1: 0.8566801061373582
False Positive Rate: 0.014690855605330429
PR-AUC: 0.938447837054874


## 9. Tune the Decision Threshold

### 9.1 Select a Threshold Under the False Positive Limit

Choose the threshold with the highest validation recall among those with a false positive rate of at most 1%. This limit is measured on validation data; it is not a guarantee for unseen data.

In [58]:
fpr, tpr, thresholds = roc_curve(
    y_val,
    y_val_prob,
    drop_intermediate=False,
)

valid_indices = np.where(fpr <= TARGET_FPR)[0]
best_idx = valid_indices[np.argmax(tpr[valid_indices])]
best_threshold = thresholds[best_idx]

print("Chosen threshold:", best_threshold)
print("Validation FPR:", fpr[best_idx])
print("Validation recall:", tpr[best_idx])

Chosen threshold: 0.9624737740077962
Validation FPR: 0.009998974645457149
Validation recall: 0.9152324272156989


### 9.2 Evaluate the Selected Threshold

In [59]:
# Classify using our selected threshold
y_val_tuned = (
    y_val_prob >= best_threshold
).astype(int)

# New confusion matrix
tn, fp, fn, tp = confusion_matrix(
    y_val,
    y_val_tuned,
    labels=[0, 1]
).ravel()

# Calculate metrics
precision = precision_score(y_val, y_val_tuned, zero_division=0)
recall = recall_score(y_val, y_val_tuned)
f1 = f1_score(y_val, y_val_tuned)

false_positive_rate = fp / (fp + tn)

print("Chosen threshold:", best_threshold)
print()
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)
print()
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("FPR:", false_positive_rate)

Chosen threshold: 0.9624737740077962

TN: 280001
FP: 2828
FN: 1054
TP: 11380

Precision: 0.8009572072072072
Recall: 0.9152324272156989
F1: 0.8542902184520682
FPR: 0.009998974645457149


### 9.3 Compare Candidate Thresholds

In [60]:
def evaluate_threshold(threshold):
    predictions = (
        y_val_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "Threshold": threshold,
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(y_val, predictions),
        "F1": f1_score(y_val, predictions),
        "FPR": fp / (fp + tn),
        "False Positives": fp,
        "Missed Attacks": fn
    }


results = []

for threshold in [0.5, 0.6, 0.7, 0.8, 0.9]:
    results.append(evaluate_threshold(threshold))

results.append(evaluate_threshold(best_threshold))

results_df = pd.DataFrame(results)

print(results_df.round(4).to_string(index=False))

 Threshold  Precision  Recall     F1    FPR  False Positives  Missed Attacks
    0.5000     0.7495  0.9997 0.8567 0.0147             4155               4
    0.6000     0.7493  0.9985 0.8561 0.0147             4153              19
    0.7000     0.7491  0.9962 0.8552 0.0147             4148              47
    0.8000     0.7497  0.9943 0.8549 0.0146             4127              71
    0.9000     0.7528  0.9879 0.8545 0.0143             4033             151
    0.9625     0.8010  0.9152 0.8543 0.0100             2828            1054


## 10. Audit Detection by Attack Family

Compare validation recall for each attack family at the default and tuned thresholds. Review missed attacks alongside the overall false positive rate to understand the threshold tradeoff.

In [61]:
# Create a validation analysis DataFrame
audit_df = val_df[["Label", "attack_cat"]].copy()

# Add the model's predicted probabilities
audit_df["attack_probability"] = y_val_prob

# Predictions using the original threshold
audit_df["pred_original"] = (
    y_val_prob >= 0.50
).astype(int)

# Predictions using the tuned threshold
audit_df["pred_tuned"] = (
    y_val_prob >= best_threshold
).astype(int)

# Only examine actual attacks
attacks_only = audit_df[
    audit_df["Label"] == 1
].copy()

# Compare detection across attack families
family_results = attacks_only.groupby(
    "attack_cat"
).agg(
    Total=("Label", "size"),
    Detected_Original=("pred_original", "sum"),
    Detected_Tuned=("pred_tuned", "sum")
)

# Calculate missed attacks
family_results["Missed_Original"] = (
    family_results["Total"]
    - family_results["Detected_Original"]
)

family_results["Missed_Tuned"] = (
    family_results["Total"]
    - family_results["Detected_Tuned"]
)

# Calculate recall for each family
family_results["Recall_Original"] = (
    family_results["Detected_Original"]
    / family_results["Total"]
)

family_results["Recall_Tuned"] = (
    family_results["Detected_Tuned"]
    / family_results["Total"]
)

print(family_results.round(4).to_string())

                Total  Detected_Original  Detected_Tuned  Missed_Original  Missed_Tuned  Recall_Original  Recall_Tuned
attack_cat                                                                                                            
Analysis          272                272             248                0            24           1.0000        0.9118
Backdoors         257                257             237                0            20           1.0000        0.9222
Dos               789                788             718                1            71           0.9987        0.9100
Exploits         4078               4077            3886                1           192           0.9998        0.9529
Fuzzers          3052               3050            2491                2           561           0.9993        0.8162
Generic          2384               2384            2353                0            31           1.0000        0.9870
Reconnaissance   1361               1361        

## 11. Compare a LightGBM Tier 1 Model

Train a second normal-vs.-attack classifier on the same splits. LightGBM uses the original numeric features and native categorical features rather than the encoded and scaled matrices used by logistic regression. This section requires `lightgbm` in the notebook environment.

### 11.1 Prepare Native Categorical Features

Copy the training and validation features, then learn each category vocabulary from training data only. Validation values outside those vocabularies become missing categorical values.

In [62]:
import lightgbm as lgb

X_train_lgb = X_train.copy()
X_val_lgb = X_val.copy()

for col in categorical_features:
    categories = X_train_lgb[col].astype("category").cat.categories
    X_train_lgb[col] = pd.Categorical(X_train_lgb[col], categories=categories)
    X_val_lgb[col] = pd.Categorical(X_val_lgb[col], categories=categories)

### 11.2 Train the Model

In [63]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=50,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)
lgb_model.fit(
    X_train_lgb,
    y_train,
    categorical_feature=categorical_features,
)

lgb_probs = lgb_model.predict_proba(X_val_lgb)[:, 1]

### 11.3 Evaluate the Default Decision Threshold

In [64]:
lgb_default_preds = (lgb_probs >= 0.50).astype(int)

print("LightGBM at threshold 0.50:")
print("Confusion matrix (rows: actual, columns: predicted; Normal, Attack):")
print(confusion_matrix(y_val, lgb_default_preds, labels=[0, 1]))
print(classification_report(
    y_val,
    lgb_default_preds,
    target_names=["Normal", "Attack"],
))

LightGBM at threshold 0.50:
Confusion matrix (rows: actual, columns: predicted; Normal, Attack):
[[281930    899]
 [  1257  11177]]
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00    282829
      Attack       0.93      0.90      0.91     12434

    accuracy                           0.99    295263
   macro avg       0.96      0.95      0.95    295263
weighted avg       0.99      0.99      0.99    295263



### 11.4 Tune the Threshold Under the False Positive Limit

Select the finite threshold with the highest validation recall subject to the same `TARGET_FPR` limit used for logistic regression. Report the resulting validation metrics; the false positive limit is not a guarantee for unseen data.

In [65]:
lgb_fpr, lgb_tpr, lgb_thresholds = roc_curve(
    y_val,
    lgb_probs,
    drop_intermediate=False,
)

lgb_valid_indices = np.where(
    (lgb_fpr <= TARGET_FPR) & np.isfinite(lgb_thresholds)
)[0]
lgb_best_idx = lgb_valid_indices[np.argmax(lgb_tpr[lgb_valid_indices])]
lgb_threshold = lgb_thresholds[lgb_best_idx]
lgb_preds = (lgb_probs >= lgb_threshold).astype(int)

lgb_tn, lgb_fp, lgb_fn, lgb_tp = confusion_matrix(
    y_val, lgb_preds, labels=[0, 1]
).ravel()

print("Chosen threshold:", lgb_threshold)
print("TN:", lgb_tn)
print("FP:", lgb_fp)
print("FN:", lgb_fn)
print("TP:", lgb_tp)
print("Precision:", precision_score(y_val, lgb_preds))
print("Recall:", recall_score(y_val, lgb_preds))
print("F1:", f1_score(y_val, lgb_preds))
print("FPR:", lgb_fp / (lgb_fp + lgb_tn))
print("Average precision:", average_precision_score(y_val, lgb_probs))

Chosen threshold: 0.2395823380876071
TN: 280002
FP: 2827
FN: 247
TP: 12187
Precision: 0.8117090715332357
Recall: 0.9801351133987454
F1: 0.888006412124745
FPR: 0.009995438940136974
Average precision: 0.9803492834684779


### 11.5 Audit Missed Attacks by Family

Measure detection and missed attacks for each family using the tuned LightGBM threshold.

In [66]:
lgb_audit = val_df[["Label", "attack_cat"]].copy()
lgb_audit["Predicted"] = lgb_preds
lgb_attacks = lgb_audit[lgb_audit["Label"] == 1].copy()

lgb_family_results = lgb_attacks.groupby("attack_cat").agg(
    Total=("Label", "size"),
    Detected=("Predicted", "sum"),
)
lgb_family_results["Missed"] = (
    lgb_family_results["Total"] - lgb_family_results["Detected"]
)
lgb_family_results["Recall"] = (
    lgb_family_results["Detected"] / lgb_family_results["Total"]
)

print(lgb_family_results.round(4).to_string())

                Total  Detected  Missed  Recall
attack_cat                                     
Analysis          272       229      43  0.8419
Backdoors         257       257       0  1.0000
Dos               789       788       1  0.9987
Exploits         4078      4068      10  0.9975
Fuzzers          3052      2863     189  0.9381
Generic          2384      2382       2  0.9992
Reconnaissance   1361      1360       1  0.9993
Shellcode         215       214       1  0.9953
Worms              26        26       0  1.0000


## 12. Train the Tier 2 Attack-Family Classifier

Train a multiclass LightGBM model using only known attacks. This evaluates family classification on actual attacks; it does not yet measure the complete two-tier pipeline with flows passed through Tier 1.

### 12.1 Prepare Attack-Only Features and Targets

In [67]:
# Reuse the attack-only splits and targets prepared in Section 5.2.
X_train_tier2_lgb = X_train_lgb.loc[attack_train.index].copy()
X_val_tier2_lgb = X_val_lgb.loc[attack_val.index].copy()

print("Tier 2 training:", X_train_tier2_lgb.shape)
print("Tier 2 validation:", X_val_tier2_lgb.shape)
print("Number of attack families:", y_train_tier2.nunique())

Tier 2 training: (58021, 36)
Tier 2 validation: (12434, 36)
Number of attack families: 9


### 12.2 Train the Multiclass Model

In [68]:
tier2_model = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)
tier2_model.fit(
    X_train_tier2_lgb,
    y_train_tier2,
    categorical_feature=categorical_features,
)

,learning_rate,0.05
,n_estimators,300
,objective,'multiclass'
,random_state,42
,n_jobs,-1
,verbosity,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,class_weight,None


### 12.3 Evaluate Validation Predictions by Attack Family

In [69]:
tier2_predictions = tier2_model.predict(X_val_tier2_lgb)

print(classification_report(
    y_val_tier2,
    tier2_predictions,
    digits=4,
    zero_division=0,
))

                precision    recall  f1-score   support

      Analysis     0.1558    0.2206    0.1826       272
     Backdoors     0.1235    0.1556    0.1377       257
           Dos     0.4056    0.2560    0.3139       789
      Exploits     0.8269    0.8705    0.8482      4078
       Fuzzers     0.9038    0.9053    0.9046      3052
       Generic     0.9024    0.8683    0.8850      2384
Reconnaissance     0.7664    0.7546    0.7605      1361
     Shellcode     0.7064    0.7163    0.7113       215
         Worms     0.4800    0.4615    0.4706        26

      accuracy                         0.7944     12434
     macro avg     0.5857    0.5788    0.5794     12434
  weighted avg     0.7949    0.7944    0.7932     12434

